# 79 — Responder A/B: top_n_for_prompt sweep (inference -> Gemini judge)

Cheap-win responder experiment (NO training): hold the validated 194 pipeline
(union+SASRec -> lgbm_clean_full -> v5-kto) fixed and vary `top_n_for_prompt`
(how many of the reranked tracks the responder sees in its prompt). Defaults to 1
-> the LM can only explain 1 of 20 tracks, capping the LLM/Gemini axis (0.30 of
the composite, currently 2.7/5). Raise to 3/5 and measure Personalization +
Explanation Quality with the offline Gemini judge (same family as the Blind-A
leaderboard) — no Blind-A submission burned.

Prereqs: GEMINI_API_KEY in Colab secrets; the Drive caches from the recall work
(sasrec_v1, dense, lgbm_clean_full). GPU runtime. Run cells in order.

The Gemini judge rubric is best-effort -> trust the RELATIVE ranking (which top_n
wins), not the absolute number.

In [ ]:
# 1) Setup — clone + HF/Gemini keys + Drive + caches + deps.
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
os.environ['USE_FLAX']='0'; os.environ['USE_TF']='0'
from google.colab import userdata, drive
os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')
os.environ['GEMINI_API_KEY']=userdata.get('GEMINI_API_KEY')   # add this Colab secret first
os.environ.setdefault('GEMINI_JUDGE_MODEL','gemini-2.5-flash')   # 1.5-flash is retired
drive.mount('/content/drive', force_remount=False)
BRANCH='recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
# Symlink the Drive caches the 194 retrieval+reranker need (same as nb74/78).
LOCAL='/content/recsys2026/experiments/cache'; os.makedirs(LOCAL, exist_ok=True)
for name,sub in [('retrieval_v2','recsys2026_retrieval_v2_cache'),('dense','recsys2026_dense_cache')]:
    src=f'/content/drive/MyDrive/{sub}'; dst=f'{LOCAL}/{name}'; os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src,dst)
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'sentence-transformers>=3.0' 'datasets' 'pandas<3.0' 'bm25s' 'lightgbm' \
    'FlagEmbedding>=1.3' 'omegaconf' 'tqdm'
!pip install -q -U google-generativeai
!pip uninstall -y torchao
print('setup done | GEMINI key present:', bool(os.environ.get('GEMINI_API_KEY')))

In [ ]:
# 2) CONFIG — derive devset top_n variants from the validated 194 baseline.
from omegaconf import OmegaConf
BASE='music-crs-baselines/config/194-union-sasrec-lgbm-cleanfull-v5kto-blindA.yaml'
TOPNS=[1,3]          # the key A/B (1=baseline, 3=plan's biggest cheap win). Add 5 for a wider sweep.
SUBSET=20            # dev sessions (~100 turns) — fast iteration; ~enough to read a top_n effect.
                     # Bump to 40+ only if the 1-vs-3 delta looks borderline.
for n in TOPNS:
    cfg=OmegaConf.load(BASE)
    cfg.test_dataset_name='talkpl-ai/TalkPlayData-Challenge-Dataset'   # DEVSET test split (so the judge re-joins context)
    cfg.top_n_for_prompt=int(n)
    tid=f'resp_top{n}_dev'
    OmegaConf.save(cfg, f'music-crs-baselines/config/{tid}.yaml')
    print(f'wrote config/{tid}.yaml  (top_n_for_prompt={n}, devset)')
print('TOPNS=',TOPNS,' SUBSET=',SUBSET)

In [ ]:
# 3) Inference per variant (SLOW: 3B responder, 320 tokens/turn + state-tracker).
#    ~SUBSET*~5 turns each; subset 40 ~= 20-50 min/variant on L4 (less on A100).
#    --batch_size 4: the 3B responder + 320 tokens OOMs at the default 16 on a 24GB L4.
#    If you still OOM, drop to 2 (and restart the runtime first to clear GPU memory).
for n in TOPNS:
    tid=f'resp_top{n}_dev'
    print('================ inference', tid, '================')
    !cd /content/recsys2026/music-crs-baselines && python -u run_inference_devset.py \
        --tid {tid} --subset {SUBSET} --batch_size 4 --device cuda --attn_implementation sdpa
print('inference done for', TOPNS)

In [ ]:
# 4) Gemini judge per variant (Personalization + Explanation, 0-5 each).
#    SLEEP=0.5 = Tier-1 (paid) pacing. Raise to ~6 if you fall back to the free tier.
SLEEP=0.5; JLIMIT=400
for n in TOPNS:
    tid=f'resp_top{n}_dev'
    print('================ judge', tid, '================')
    !cd /content/recsys2026/music-crs-baselines && python -u ../scripts/gemini_judge_responses.py \
        --tid {tid} --limit {JLIMIT} --sleep {SLEEP}

In [ ]:
# 5) Compare (relative ranking is what matters). Reads each judge JSON; falls
#    back to the per-run means printed in cell 4 if the schema differs.
import json, os, statistics as st
print('=== responder top_n A/B (Gemini judge, devset) ===')
print(f"{'top_n':>6} {'n':>5} {'personalization':>16} {'explanation':>12} {'mean':>7}")
for n in TOPNS:
    tid=f'resp_top{n}_dev'; p=f'music-crs-baselines/exp/judge/{tid}_gemini.json'
    if not os.path.exists(p): print(f'{n:>6}  (no judge output — check cell 4)'); continue
    obj=json.load(open(p)); rows=obj if isinstance(obj,list) else obj.get('rows',[])
    ps=[r['personalization'] for r in rows if isinstance(r,dict) and r.get('personalization') is not None]
    es=[r['explanation_quality'] for r in rows if isinstance(r,dict) and r.get('explanation_quality') is not None]
    if ps:
        pm,em=st.mean(ps),st.mean(es)
        print(f'{n:>6} {len(ps):>5} {pm:>16.3f} {em:>12.3f} {(pm+em)/2:>7.3f}')
    else:
        print(f'{n:>6}  (means in cell-4 output)')